In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from ppfn.dataset.prior.bnn_prior import BNNPrior
from ppfn.dataset.prior.bnn_link_fn import VectorizedParameterLinker

In [ ]:
def plot_bnn_prior_surfaces(num_samples=4, num_hp=40, num_fid=60):
    """
    Generates a grid of 3D plots showing the HP-Fidelity-Performance surface
    for different random draws from the BNN Prior.
    """
    # 1. Setup (Assumes BNNPrior and VectorizedParameterLinker are already defined/imported)
    num_inputs = 1
    num_outputs = 23
    prior = BNNPrior(num_inputs, num_outputs)
    linker = VectorizedParameterLinker(prior)

    # 2. Define Grids
    hp_axis = np.linspace(0, 1, num_hp)
    fid_axis = np.linspace(0.01, 1.0, num_fid)
    HP, FID = np.meshgrid(hp_axis, fid_axis)

    # 3. Setup Plotting Layout
    cols = 2
    rows = (num_samples + 1) // 2
    fig = plt.figure(figsize=(16, 6 * rows))

    for k in range(num_samples):
        # Sample a new BNN "universe"
        mlp = prior.sample()
        mlp.eval()

        # Generate parameters across the HP range
        hp_tensor = torch.from_numpy(hp_axis).float().view(-1, 1)
        with torch.no_grad():
            bnn_out = mlp(hp_tensor).numpy()

        # Link to curve parameters (using typical y-bounds)
        curve_fn = linker.curve_factory(bnn_out, y0=0.1, ymax=0.95)

        # Evaluate surface: Shape (num_fid, num_hp)
        # We evaluate the curve for every HP config (cid) across all fidelity points
        Z = np.zeros((num_fid, num_hp))
        for i in range(num_hp):
            # cid=i selects the parameters for the i-th hyperparameter value
            Z[:, i] = curve_fn(fid_axis, cid=i)

        # 4. 3D Visualization
        ax = fig.add_subplot(rows, cols, k + 1, projection="3d")
        ax.plot_surface(
            HP,
            FID,
            Z,
            cmap="turbo",
            antialiased=True,
            alpha=0.9,
            rcount=num_fid,
            ccount=num_hp,
        )

        ax.set_title(f"BNN Prior Sample {k + 1}", fontsize=14, pad=20)
        ax.set_xlabel("Hyperparameter")
        ax.set_ylabel("Fidelity")
        ax.set_zlabel("Performance (y)")
        ax.set_zlim(0, 1)
        ax.view_init(elev=25, azim=-130)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_bnn_prior_surfaces(num_samples=100)